# 기상청_지상(종관, ASOS) 일자료 조회서비스
이전 hour 용 다운로드 코드를 기반으로 daily (일자료) 다운로드 처리를 수행하는 코드입니다.

In [4]:
import requests
import pandas as pd
import time
import os
import glob
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def fetch_resilient_daily_weather_data():
    # 1. 초기 설정
    url = 'http://apis.data.go.kr/1360000/AsosDalyInfoService/getWthrDataList'
    # Decoding 인증키 사용 (requests가 자동 인코딩하므로 == 형태가 안전)
    service_key = 'gR9efoM90FwF0PklBCvwsoDUOCQy8FMzFbsJLI8ARdJHTCPCD32vV40mNCVXUDtO0CjcDfd8rgHQZcMSxhOcmg=='
    
    start_year = 20
    end_year = 2025
    temp_dir = 'asos_daily_raw_data'
    if not os.path.exists(temp_dir): os.makedirs(temp_dir)

    stn_ids = [
        90, 95, 98, 99, 100, 101, 102, 104, 105, 106, 108, 112, 114, 115, 119, 121, 
        127, 129, 130, 131, 133, 135, 136, 137, 138, 140, 143, 146, 152, 155, 156, 
        159, 162, 165, 168, 169, 170, 172, 174, 184, 185, 188, 189, 192, 201, 202, 
        203, 211, 212, 216, 217, 221, 226, 232, 235, 236, 238, 243, 244, 245, 247, 
        248, 251, 252, 253, 254, 255, 257, 258, 259, 260, 261, 262, 263, 264, 266, 
        271, 272, 273, 276, 277, 278, 279, 281, 283, 284, 285, 288, 289, 294, 295
    ]

    # 세션 및 재시도 설정 (타임아웃 방지 핵심)
    session = requests.Session()
    retry = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    session.mount('http://', HTTPAdapter(max_retries=retry))

    for stn_id in stn_ids:
        stn_file_path = os.path.join(temp_dir, f"STN_{stn_id}_daily.csv")
        if os.path.exists(stn_file_path):
            continue

        print(f"\n[지점 {stn_id}] 수집 시작...")
        stn_frames = []
        
        for year in range(start_year, end_year + 1):
            page_no = 1
            while True:
                params = {
                    'serviceKey': service_key, 'pageNo': str(page_no), 'numOfRows': '700',
                    'dataType': 'JSON', 'dataCd': 'ASOS', 'dateCd': 'DAY',
                    'startDt': f'{year}0101', 'endDt': f'{year}1231',
                    'stnIds': str(stn_id)
                }

                try:
                    # timeout을 60초로 늘리고 세션 사용
                    response = session.get(url, params=params, timeout=60)
                    res_json = response.json()
                    header = res_json.get('response', {}).get('header', {})

                    if header.get('resultCode') == '00':
                        items = res_json.get('response', {}).get('body', {}).get('items', {}).get('item', [])
                        if items:
                            stn_frames.append(pd.DataFrame(items))
                            if len(items) < 700: break
                            page_no += 1
                            time.sleep(0.2) # 속도를 조금 늦춰 서버 차단 방지
                        else: break
                    else:
                        print(f"  - {year}년 오류: {header.get('resultMsg')}")
                        break
                except requests.exceptions.Timeout:
                    print(f"  - [타임아웃] {year}년 {page_no}P 재시도 중...")
                    time.sleep(5) # 타임아웃 시 잠시 대기
                    continue 
                except Exception as e:
                    print(f"  - 예외: {e}")
                    break
        
        if stn_frames:
            pd.concat(stn_frames, ignore_index=True).to_csv(stn_file_path, index=False, encoding='utf-8-sig')
            print(f"  - 지점 {stn_id} 저장 완료")

    # 통합 로직
    print("\n최종 파일 통합 중...")
    all_files = glob.glob(os.path.join(temp_dir, "STN_*_daily.csv"))
    if all_files:
        pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True).to_csv(f"ASOS_DAILY_TOTAL_{start_year}_{end_year}.csv", index=False, encoding='utf-8-sig')
        print("모든 작업 완료!")

if __name__ == "__main__":
    fetch_resilient_daily_weather_data()


[지점 90] 수집 시작...
  - 지점 90 저장 완료

[지점 95] 수집 시작...
  - 1974년 오류: NO_DATA
  - 1975년 오류: NO_DATA
  - 1976년 오류: NO_DATA
  - 1977년 오류: NO_DATA
  - 1978년 오류: NO_DATA
  - 1979년 오류: NO_DATA
  - 1980년 오류: NO_DATA
  - 1981년 오류: NO_DATA
  - 1982년 오류: NO_DATA
  - 1983년 오류: NO_DATA
  - 1984년 오류: NO_DATA
  - 1985년 오류: NO_DATA
  - 1986년 오류: NO_DATA
  - 1987년 오류: NO_DATA
  - 지점 95 저장 완료

[지점 98] 수집 시작...
  - 1974년 오류: NO_DATA
  - 1975년 오류: NO_DATA
  - 1976년 오류: NO_DATA
  - 1977년 오류: NO_DATA
  - 1978년 오류: NO_DATA
  - 1979년 오류: NO_DATA
  - 1980년 오류: NO_DATA
  - 1981년 오류: NO_DATA
  - 1982년 오류: NO_DATA
  - 1983년 오류: NO_DATA
  - 1984년 오류: NO_DATA
  - 1985년 오류: NO_DATA
  - 1986년 오류: NO_DATA
  - 1987년 오류: NO_DATA
  - 1988년 오류: NO_DATA
  - 1989년 오류: NO_DATA
  - 1990년 오류: NO_DATA
  - 1991년 오류: NO_DATA
  - 1992년 오류: NO_DATA
  - 1993년 오류: NO_DATA
  - 1994년 오류: NO_DATA
  - 1995년 오류: NO_DATA
  - 1996년 오류: NO_DATA
  - 1997년 오류: NO_DATA
  - 지점 98 저장 완료

[지점 99] 수집 시작...
  - 1974년 오류: NO_DATA
  - 1975년 오류: NO_DATA
